# Week 6 — Selection Effects, Completeness and Survey Population Inference

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../06_Bayesian/Bayesian_3.ipynb`
- `../../07_MCMC/MCMC.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Distinguish censoring, truncation, missingness and survey selection.
- Derive the likelihood for a population observed through a selection function.
- Explain the role of the selection normalisation or detection efficiency.
- Use injection/recovery or simulation to estimate completeness.
- Demonstrate how ignoring selection biases binary-SMBH population inference.

## 1. The observed catalogue is conditional on detection


Astronomical catalogues are not random samples from the Universe. Detection depends on flux, variability, cadence, sky position, signal-to-noise and pipeline decisions. If $\theta$ denotes latent source parameters and $\Lambda$ population hyperparameters, then an intrinsic density $p(\theta\mid\Lambda)$ becomes a detected density

$$
p(\theta\mid{\rm det},\Lambda)
=
\frac{p_{\rm det}(\theta)\,p(\theta\mid\Lambda)}
{\alpha(\Lambda)},
$$

where

$$
\alpha(\Lambda)
=
\int p_{\rm det}(\theta)\,p(\theta\mid\Lambda)\,d\theta
$$

is the population-averaged detection efficiency.

The denominator is not optional. It changes as $\Lambda$ changes, so omitting it generally biases population inference.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Create a synthetic selection function

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, sys, matplotlib.pyplot as plt
sys.path.append('../src')
from course_utils import load_scalar_sample
df=load_scalar_sample()
z=df['inputs__redshift'].to_numpy()
logM=np.log10(df['inputs__total_mass'].to_numpy())
eta = 1.0 - 0.9*(z-1.0) + 0.7*(logM-8.0)
pdet = 1/(1+np.exp(-eta))
rng=np.random.default_rng(2026)
det=rng.random(len(df))<pdet
print('detection fraction:',det.mean())

## 2. Truncation, censoring and missingness are different


In **truncation**, objects outside the observable region never enter the dataset. In **censoring**, an object is known to exist but only a limit is recorded. Missingness can occur for many reasons and is not automatically random.

For example, a non-detection in one LSST band is different from a source that never entered the sample because it failed a multi-band selection. The likelihood must match the data-generating and reporting process.

Survival-analysis methods can handle some censoring structures, while explicit selection functions are needed for truncated survey populations.

### Worked computational demonstration — Show induced bias

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
fig,ax=plt.subplots()
ax.hist(z,bins=35,density=True,histtype='step',label='intrinsic')
ax.hist(z[det],bins=35,density=True,histtype='step',label='detected')
ax.set_xlabel('redshift'); ax.legend(); plt.show()
print('intrinsic median z',np.median(z))
print('detected median z',np.median(z[det]))

## 3. Selection-corrected population likelihood


For detected objects with data $D_i$,

$$
p(\{D_i\}\mid\Lambda,{\rm det})
\propto
\prod_i
\frac{\int p(D_i\mid\theta_i)p(\theta_i\mid\Lambda)\,d\theta_i}
{\alpha(\Lambda)}.
$$

Depending on whether the total event count is modelled, additional Poisson-rate terms may appear. The key conceptual point is that the model must account for the part of the population that could have been present but was not observed.

This is where simulations become operationally useful. If $p_{\rm det}$ is difficult to write analytically, injection/recovery can estimate the fraction of simulated sources recovered as a function of relevant parameters.

### Worked computational demonstration — Importance-weighting diagnostic

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
w=1/np.clip(pdet[det],0.05,1)
weighted_mean=np.sum(w*z[det])/np.sum(w)
print('intrinsic mean z:',z.mean())
print('naive detected mean:',z[det].mean())
print('IPW diagnostic mean:',weighted_mean)

## 4. A Rubin-like pedagogical selection function


For the course we can create a synthetic selection rule rather than pretending to reproduce the full Rubin pipeline. For example,

$$
{\rm logit}\,p_{\rm det}
=
a+b_1(z-z_0)+b_2(\log M-M_0)
+b_3(\mathrm{variability\ S/N}-s_0).
$$

The purpose is not survey realism; it is to show how a plausible dependence reshapes the observed $M,q,P,z$ distributions. Students should compare naive estimates from the detected sample with selection-aware estimates.

The same logic applies to any later ML training sample: if the training catalogue is selected differently from deployment data, the model learns the selected distribution.

## 5. Diagnostics for selection models


Selection functions should be validated just like physical models. Plot detection fraction against the variables used to define it, examine multidimensional interactions, and check whether reweighted or forward-simulated catalogues reproduce the detected sample.

Inverse-probability weights $1/p_{\rm det}$ can be useful for intuition, but become unstable when detection probabilities approach zero. Full generative modelling is preferable when extrapolation into weakly observed regions matters.

A critical scientific conclusion is often that a region is **not identifiable** from the available survey, rather than that a correction can recover it perfectly.

## Deep dive — shape, rate and selection normalisation

There are two related population questions: the **shape** of the distribution and the **absolute rate/number** of sources. If the analysis conditions on observing $N$ detected objects, the selection-normalised shape likelihood contains powers of $\alpha(\Lambda)$. If the total number itself is informative, a Poisson point-process formulation also models the expected number of detections.

A schematic Poisson-process likelihood is

$$
\mathcal L(\Lambda)
\propto
e^{-\mu_{\rm det}(\Lambda)}
\prod_{i=1}^{N}
\int p(D_i\mid\theta)
\,\lambda(\theta\mid\Lambda)
\,d\theta,
$$

where $\lambda$ is a population intensity and

$$
\mu_{\rm det}(\Lambda)
=
\int p_{\rm det}(\theta)\lambda(\theta\mid\Lambda)d\theta.
$$

This formulation shows why selection affects both the distribution and the expected count.

### Injection/recovery

Real selection functions are often measured by injecting synthetic sources into a pipeline and recording recovery. The estimated detection probability may then be represented by bins, splines, logistic models or flexible classifiers.

However, the selection model itself has uncertainty. A finite injection campaign does not give exact $p_{\rm det}$. At advanced levels, uncertainty in the selection model should be propagated rather than treated as known.

### Positivity and overlap

Inverse-probability approaches rely on nonzero detection probability in relevant regions. If a region of parameter space is never observed, the intrinsic population there cannot be recovered from the detected sample without extrapolative assumptions. This is an identifiability limitation, not merely a need for larger weights.

Students should mark regions of weak overlap explicitly rather than presenting corrected estimates as equally reliable everywhere.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Loredo-style hierarchical population inference literature.
- Hogg, Bovy & Lang on data analysis likelihood thinking.
- Crete Bayesian/MCMC notebooks as computational prerequisites.

# Part II — Extended lecture development

## 13. Deriving the detected-population density step by step

Start with the joint probability

$$
p(\theta,{\rm det}\mid\Lambda)
=
p({\rm det}\mid\theta,\Lambda)
p(\theta\mid\Lambda).
$$

If detection depends on $\theta$ but not directly on $\Lambda$ once
$\theta$ is known,

$$
p({\rm det}\mid\theta,\Lambda)=p_{\rm det}(\theta).
$$

Condition on detection:

$$
p(\theta\mid{\rm det},\Lambda)
=
\frac{
p(\theta,{\rm det}\mid\Lambda)
}{
p({\rm det}\mid\Lambda)
}.
$$

The denominator is

$$
p({\rm det}\mid\Lambda)
=
\int
p_{\rm det}(\theta)
p(\theta\mid\Lambda)d\theta
\equiv\alpha(\Lambda).
$$

This derivation is worth doing on the board. It shows exactly where the
selection normalisation comes from.

## 14. Selection on observables rather than latent truth

Real detection algorithms operate on observed quantities. If $D$ is noisy
data,

$$
P({\rm det}\mid\theta)
=
\int
P({\rm det}\mid D)
p(D\mid\theta)dD.
$$

Near a detection threshold, noise scatters sources across the boundary. Thus a
hard observed-flux cut becomes a smooth latent-parameter selection probability.

This is the statistical origin of Eddington-like threshold biases: the number
of sources scattered into a selected region need not equal the number scattered
out when the underlying population density is steep.

## 15. Detection efficiency and the population model are coupled

The quantity

$$
\alpha(\Lambda)
=
\int p_{\rm det}(\theta)p(\theta\mid\Lambda)d\theta
$$

changes when $\Lambda$ changes.

For example, if one population model places more probability at high redshift,
and detection efficiency declines with redshift, its average detection
efficiency is lower.

This is why a single fixed "completeness correction" independent of the
population model can be inadequate for hierarchical inference.

## 16. Monte Carlo calculation of selection normalisation

When the integral for $\alpha(\Lambda)$ is unavailable analytically, Monte
Carlo can estimate it.

If

$$
\theta_j\sim p(\theta\mid\Lambda),
$$

then

$$
\hat\alpha(\Lambda)
=
\frac1N\sum_j p_{\rm det}(\theta_j).
$$

If simulations were drawn from a different proposal distribution $q(\theta)$,
importance weights can reweight them:

$$
\alpha(\Lambda)
=
E_q\left[
p_{\rm det}(\theta)
\frac{p(\theta\mid\Lambda)}{q(\theta)}
\right].
$$

This is closely related to simulation-based population inference.

In [ ]:
# Monte Carlo estimate of detection efficiency for a toy population
rng=np.random.default_rng(2026)

def pdet_z(z):
    return 1/(1+np.exp(1.5*(z-2.0)))

# hypothetical population z ~ Gamma(shape, scale)
def alpha_mc(shape,scale,N=100_000):
    z=rng.gamma(shape,scale,N)
    return pdet_z(z).mean()

for pars in [(2,.5),(2,1.0),(3,1.0)]:
    print(pars,alpha_mc(*pars))

## 17. Effective sample size under inverse weights

For weights $w_i$, a common effective-sample-size diagnostic is

$$
N_{\rm eff}
=
\frac{(\sum_i w_i)^2}
{\sum_i w_i^2}.
$$

If a few inverse-probability weights dominate, $N_{\rm eff}$ can be tiny even
when the catalogue contains thousands of sources.

This is a quantitative way to show students that selection correction cannot
create information where the survey has almost none.

In [ ]:
w=1/np.clip(pdet[det],0.01,1)
neff=(w.sum()**2)/(w@w)
print("detected rows:",len(w))
print("weighted effective N:",neff)

## 18. Selection functions can depend on context

A more realistic detection probability is

$$
p_{\rm det}(\theta,c),
$$

where $c$ can include field, cadence, depth or observing conditions.

The provider data document WFD/DDF labels and actual time arrays. The course
does not invent additional cadence assumptions; instead, students can measure
empirical cadence properties directly from the supplied light curves.

This supports a useful comparison:
- selection conditional on field;
- population differences conditional on field;
- whether field changes only observability or is accidentally associated with
  simulated physical inputs.

## 19. Selection and classification thresholds

A machine-learning probability threshold is itself a selection rule.

If a classifier sends only $p>0.9$ sources to follow-up, the resulting
spectroscopic sample has selection probability determined by the classifier,
its feature distribution and the threshold.

Future population inference from that follow-up sample must account for this
adaptive selection.

Thus catalog construction, ML decision thresholds and follow-up policies all
belong to the same statistical framework.

## 20. Non-detections contain information

Discarding non-detections can waste information. If an object is known to have
an upper limit $F<F_{\rm lim}$, the likelihood contribution is

$$
P(F_{\rm obs}<F_{\rm lim}\mid\theta)
=
\int_{-\infty}^{F_{\rm lim}}
p(F_{\rm obs}\mid\theta)dF_{\rm obs}.
$$

This is a censored measurement, not a missing source.

Upper limits can constrain models strongly, particularly when a model predicts
that a source should have been easily detectable.

## 21. Selection-corrected reporting

A strong survey-population paper should state:
- parent population definition;
- detection/selection rule;
- how completeness was estimated;
- whether rate or only shape is inferred;
- where $p_{\rm det}$ is small;
- whether uncertainty in the selection function is propagated;
- sensitivity to selection-model alternatives.

These items should become part of students' standard statistical checklist.

# Part III — Worked selection problems

## Problem A — a two-bin selection bias

Suppose half the intrinsic population is low redshift and half high redshift.
Detection probabilities are 0.8 and 0.2 respectively.

Among detected objects, expected proportions are proportional to

$$
0.5\times0.8=0.4
$$

and

$$
0.5\times0.2=0.1.
$$

Normalising gives 80% low-redshift and 20% high-redshift detections.

Thus a 50/50 intrinsic population becomes an 80/20 observed catalogue without
any physical evolution. This simple arithmetic should be mastered before the
continuous selection likelihood.

## Problem B — zero completeness

If $p_{\rm det}=0$ for $z>5$, can the intrinsic $z>5$ population fraction
be recovered from detected data alone?

No. The likelihood contains no direct information from that region. Any answer
comes from extrapolation or prior/model assumptions.

This is a clean example of non-identifiability created by the observation
process.

## Problem C — why weighting can explode

If a detected source has $p_{\rm det}=0.01$, inverse weighting gives $w=100$.
One source can then represent roughly one hundred intrinsic sources.

A handful of such detections create enormous estimator variance. Ask students
to compare nominal catalogue size with weighted effective sample size before
trusting an inverse-weighted result.